# House PTR extraction coverage audit

## tl;dr

The frozen network is **known to be incomplete**. All 515 indexed 2025 PTRs and 418 indexed 2026 PTRs have checkpoints, but 66 and 52 respectively have zero extracted transactions. Those 118 PDFs are image-only, with 852 pages in total. A visually reviewed 2025 Khanna filing contains marked common-stock purchases absent from extraction. The accepted network remains a 55-reporter partial extract.

This notebook audits coverage; it does not recover transactions or certify the remaining records.

## Context & Methods

Unit: one indexed House PTR filing, keyed by filing year and filing ID. Compare the live Clerk annual indexes with the frozen upstream index and parser checkpoints, then inspect every zero-row PDF for a usable text layer. Original PDF bytes are hashed. Visually inspect selected pages to test whether missing text means missing purchases.

### Key Assumptions

A checkpoint is processing evidence, not proof of correct extraction. A 2026 filing can contain 2025 transactions; its filing year alone does not determine inclusion. A zero-row PDF can contain sales only, so missing qualifying stock purchases require source review. Tables suffice for this bounded coverage check; a chart adds no information.

Source: [House Clerk disclosures](https://disclosures-clerk.house.gov/FinancialDisclosure), frozen scraper commit `a064720e90e715f516cd2a725ece0b6f7a73632b`. The JSON companion preserves official index URLs and hashes.

## Data

Place `filing-coverage.csv` and `filing-coverage.json` beside this notebook, or run it from the repository validation directory. Empty PDF fields mean the PDF text layer was not inspected; they do not mean zero.

In [1]:
from pathlib import Path
import csv, json, hashlib
from collections import Counter
base = Path.cwd()
with (base / 'filing-coverage.csv').open(encoding='utf-8', newline='') as f:
    filings = list(csv.DictReader(f))
summary = json.loads((base / 'filing-coverage.json').read_text(encoding='utf-8'))
assert hashlib.sha256((base / 'filing-coverage.csv').read_bytes()).hexdigest() == summary['audit_inventory_sha256']
assert len({(r['filing_year'], r['filing_id']) for r in filings}) == len(filings)
print(f'{len(filings):,} distinct indexed filings in the recorded inventory.')

933 distinct indexed filings in the recorded inventory.


## Results

### Extraction coverage by filing year

In [2]:
for year in ('2025', '2026'):
    records = [r for r in filings if r['filing_year'] == year]
    zero = [r for r in records if r['parsed_transaction_rows'] == '0']
    no_text = [r for r in zero if r['pdf_text_characters'] == '0']
    missing = [r for r in records if r['parser_status'] == 'missing_checkpoint']
    print(f'{year}: {len(records)} indexed; {len(missing)} missing checkpoints; {len(zero)} zero-row filings ({len(zero)/len(records):.1%}); {len(no_text)} confirmed without text; {sum(int(r["pdf_pages"]) for r in no_text)} pages.')
    assert len(no_text) == len(zero)
    assert all(r['parser_status'] == 'needs_fallback' for r in zero)
    assert not missing

2025: 515 indexed; 0 missing checkpoints; 66 zero-row filings (12.8%); 66 confirmed without text; 451 pages.
2026: 418 indexed; 0 missing checkpoints; 52 zero-row filings (12.4%); 52 confirmed without text; 401 pages.


### Confirmed omitted purchase evidence

[Original Khanna filing, 8220750](https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/8220750.pdf). Page 1 identifies the reporter. Pages 3, 10 and 20 show checked purchase columns, common-stock asset names, 2025 transaction dates and DC/SP owner codes. These examples were manually read from the rendered pages; they are not automatically resolved ticker rows and have not been inserted into the network.

In [3]:
case = summary['confirmed_missing_filing']
filing = next(r for r in filings if r['filing_year'] == '2025' and r['filing_id'] == case['filing_id'])
assert filing['parsed_transaction_rows'] == '0'
assert filing['pdf_text_characters'] == '0'
for row in case['examples']:
    print(f'Page {row["page"]}: {row["asset"]} | {row["transaction_type"]} | {row["transaction_date"]} | owner {row["owner_code"]}')

Page 3: GENERAL MILLS INC CMN | Purchase | 2025-01-17 | owner DC
Page 3: GENERAL MILLS INC CMN | Purchase | 2025-01-21 | owner DC
Page 10: DOW INC CMN | Purchase | 2025-01-22 | owner DC
Page 20: A.O. SMITH CORP (DEL) CMN | Purchase | 2025-01-17 | owner SP


## Takeaways

**High severity:** Missing scanned filings bias the member population, stock overlap and party mixing. Archive completeness is not transaction completeness. Exact missing purchase and member counts remain unknown; 55 is the size of the accepted extract, not the total number of House stock purchasers.

**Remediation:** Recover all flagged scanned filings using OCR and checkbox-aware table extraction; verify asset names, transaction type, dates, owner and amounts against source pages; resolve symbols and reconcile amendments before publishing a new accepted extract. Restrict recovered records to transaction dates in 2025, retain original PDF/page provenance, then rerun every graph calculation. Check nonzero parsed filings for page/row omissions separately.

The site now names the graph types and labels the data as a partial extraction. Current records remain frozen while missing rows are reviewed.